# Qwen3-Embedding-0.6B for Module 5 Semantic Router

Notebook này được thiết kế để chạy trên Kaggle (GPU T4x2 hoặc L4).
Nó sẽ đọc `physical_graph.json`, nhúng các văn bản bằng `Qwen/Qwen3-Embedding-0.6B` và xuất 3 file đầu ra cần thiết cho M5.

In [ ]:
!pip install -U sentence-transformers numpy

In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import os

# Đổi đường dẫn này phù hợp với thư mục dataset bạn upload trên Kaggle
INPUT_GRAPH_PATH = "/kaggle/input/physical-graph/physical_graph.json"
OUTPUT_DIR = "/kaggle/working/"

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
TASK_DESCRIPTION = "Given a legal clause from Vietnamese law, extract its embedding for semantic routing and finding legal exceptions or conditions."
BATCH_SIZE = 32

In [ ]:
print(f"Đang nạp graph từ {INPUT_GRAPH_PATH}...")
with open(INPUT_GRAPH_PATH, "r", encoding="utf-8") as f:
    graph = json.load(f)

nodes = graph.get("nodes", [])
print(f"Tổng số nodes: {len(nodes)}")

node_ids = []
texts = []

for n in nodes:
    node_ids.append(n["id"])
    # Lấy text từ properties
    text = n.get("properties", {}).get("text", "")
    texts.append(text)


In [ ]:
print(f"Đang tải model {MODEL_NAME}...")
# Model card Qwen3-Embedding-0.6B hỗ trợ sentence-transformers natively
model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

print("Bắt đầu embedding...")
# Thêm task_description (instruction) theo đúng API của instructor/Qwen3
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    prompt=TASK_DESCRIPTION
)

print(f"Embedding shape: {embeddings.shape}")

In [ ]:
print("Đang lưu kết quả...")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Save embeddings.npy
np.save(os.path.join(OUTPUT_DIR, "embeddings.npy"), embeddings)

# 2. Save node_ids.json
with open(os.path.join(OUTPUT_DIR, "node_ids.json"), "w", encoding="utf-8") as f:
    json.dump(node_ids, f, ensure_ascii=False, indent=2)

# 3. Save embedding_config.json
config = {
    "model_name": MODEL_NAME,
    "task_description": TASK_DESCRIPTION,
    "embedding_dim": embeddings.shape[1],
    "num_nodes": embeddings.shape[0]
}
with open(os.path.join(OUTPUT_DIR, "embedding_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Hoàn thành! Bạn có thể tải 3 file từ Output của Kaggle Notebook về local.")